In [9]:
import pandas as pd
import csv
import pickle
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer

In [10]:
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-german-uncased")

In [11]:
path_train = "Input/Data/train/"
path_test = "Input/Data/test/"

train_data = pd.read_json(path_train + "test_task1.json")
train_features = pd.read_csv(path_train + "test_features.csv")
test_data = pd.read_csv(path_test + "comments_extended.csv")
test_features = pd.read_csv(path_test + "features.csv")

In [12]:
path = "Input/Data/"

with open(path + "positive_words.csv", newline='', encoding='utf-8') as infile:
    positive_words = [w for [w] in csv.reader(infile)]

with open(path + "positive_tokens.pkl", "rb") as infile: 
   positive_tokens = pickle.load(infile)

In [13]:
# adding relevant features to features.csv for test data

sentiment_orig = []
sentiment_spelling_corrected = []
sentiment_translated = []
sentiment_anger = []
sentiment_disgust = []
sentiment_fear = []
sentiment_joy = []
sentiment_neutral = []
sentiment_sadness = []
sentiment_surprise = []
flausch_words = []
flausch_tokens = []
flausch_ratio = []

for i in range(len(test_features)):

    comment = test_data.iloc[i]["comment"]
    tokens = tokenizer.tokenize(comment)

    # add sentiments
    sentiment_orig.append(test_data.iloc[i]["sentiment_orig"])
    sentiment_spelling_corrected.append(test_data.iloc[i]["sentiment_spelling_corrected"])
    sentiment_translated.append(test_data.iloc[i]["sentiment_translated"])
    sentiment_anger.append(test_data.iloc[i]["sentiment_anger"])
    sentiment_disgust.append(test_data.iloc[i]["sentiment_disgust"])
    sentiment_fear.append(test_data.iloc[i]["sentiment_fear"])
    sentiment_joy.append(test_data.iloc[i]["sentiment_joy"])
    sentiment_neutral.append(test_data.iloc[i]["sentiment_neutral"])
    sentiment_sadness.append(test_data.iloc[i]["sentiment_sadness"])
    sentiment_surprise.append(test_data.iloc[i]["sentiment_surprise"])

    # add word count
    flausch_word_count = 0
    for w in positive_words:
        if w in comment:
            flausch_word_count += 1
    flausch_words.append(flausch_word_count)

    # add token count
    flausch_token_count = 0
    for t in tokens:
        if t in positive_tokens:
            flausch_token_count += 1
    flausch_tokens.append(flausch_token_count)
    # positive to all token ratio
    flausch_ratio.append(flausch_token_count/len(tokens))

test_features["sentiment_orig"] = sentiment_orig
test_features["sentiment_spelling_corrected"] = sentiment_spelling_corrected
test_features["sentiment_translated"] = sentiment_translated
test_features["sentiment_anger"] = sentiment_anger
test_features["sentiment_disgust"] = sentiment_disgust
test_features["sentiment_fear"] = sentiment_fear
test_features["sentiment_joy"] = sentiment_joy
test_features["sentiment_neutral"] = sentiment_neutral
test_features["sentiment_sadness"] = sentiment_sadness
test_features["sentiment_surprise"] = sentiment_surprise
test_features["flausch_words"] = flausch_words
test_features["flausch_tokens"] = flausch_tokens
test_features["flausch_ratio"] = flausch_ratio

test_features.to_csv(path_test + "features.csv", index=False)

In [28]:
def get01(flausch):
    if flausch == "yes":
        return 1
    else:
        return 0

In [29]:
def get_yes_no(x):
    if x == 0:
        return "no"
    else:
        return "yes"

In [35]:
# fit logistic regression to training data

X_train = []
y_train = []

for i in range(len(train_features)):

    id = train_features.iloc[i]["id"]
    flausch = train_data[train_data["id"] == id]["flausch"].item()
    y_train.append(get01(flausch))

    feature_vec = []
    for c in [
            "gbert-large_yes_comment",
            "sentiment_orig",
            "sentiment_spelling_corrected",
            "sentiment_translated",
            "sentiment_anger", 
            "sentiment_disgust",
            "sentiment_fear", 
            "sentiment_joy", 
            "sentiment_neutral",
            "sentiment_sadness",
            "sentiment_surprise",
            "flausch_words",
            "flausch_tokens",
            "flausch_ratio"
            ]: 
        feature_vec.append(train_features.iloc[i][c])
    X_train.append(feature_vec)

In [36]:
# get vectors for test data

X_test = []

for i in range(len(test_features)):

    feature_vec = []
    for c in [
            "gbert-large_yes_comment",
            "sentiment_orig",
            "sentiment_spelling_corrected",
            "sentiment_translated",
            "sentiment_anger", 
            "sentiment_disgust",
            "sentiment_fear", 
            "sentiment_joy", 
            "sentiment_neutral",
            "sentiment_sadness",
            "sentiment_surprise",
            "flausch_words",
            "flausch_tokens",
            "flausch_ratio"
            ]: 
        feature_vec.append(test_features.iloc[i][c])
    X_test.append(feature_vec)

In [37]:
log_reg = LogisticRegression().fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
pred = [get_yes_no(y) for y in y_pred]

In [40]:
# put results into file for submisison

predictions = test_data[["document", "comment_id"]].copy()
predictions["flausch"] = pred
predictions.to_csv("predictions/task1-predicted.csv", index=False)